# Your model vs BaratiLab baseline — physical-space comparison

Both reconstruct the **same test set** (seqs 36-39) from the **identical** sparse+NN-fill input toward the
**identical** ground truth (verified: your denormalized GT == BaratiLab `reference`, max diff 0). The only
difference is the model. Your results are stored normalized, so they are **denormalized x4.7988** to physical
vorticity units to match BaratiLab (which is already physical).

Note: physical MSE = normalized MSE x std^2, so the model *ranking* is identical in either space; physical is
shown here because it is interpretable and matches how BaratiLab reports. Select **Python (venv-ddpm)**, Run All.

In [ ]:
import os, pickle
import numpy as np
import matplotlib.pyplot as plt

STD = 4.7988   # your model's training normalization (kf_2d first-32) -> denormalize to physical

ROOT = os.getcwd()
while ROOT != "/" and not os.path.isdir(os.path.join(ROOT, "monitoring")):
    ROOT = os.path.dirname(ROOT)
RDIR = os.path.join(ROOT, "monitoring", "sequence_reconstructions")

b = pickle.load(open(os.path.join(ROOT, "base_results", "baseline", "baratilab_results.pkl"), "rb"))
ref, inp, pred_b = b["reference"], b["input"], b["pred_it2"]   # physical, (4,318,256,256,3)
SEQS = [36, 37, 38, 39]

def mine_final(seq):
    fr = pickle.load(open(os.path.join(RDIR, f"sequence_reconstruction_seq{seq}.pkl"), "rb"))["frames"]
    return np.stack([f["final"] for f in fr]).astype(np.float32) * STD

pred_m = np.stack([mine_final(s) for s in SEQS])   # (4,318,256,256,3) physical
print("loaded | reference", ref.shape, "| mine", pred_m.shape, "| units: physical vorticity")

In [ ]:
# --- physical MSE / RMSE per sequence + overall ---
mse_m = ((ref - pred_m) ** 2).mean(axis=(2, 3, 4))   # (4, 318) per-seq per-frame
mse_b = ((ref - pred_b) ** 2).mean(axis=(2, 3, 4))
print(f"{'seq':>5}{'mine MSE':>11}{'barati MSE':>12}{'mine RMSE':>11}{'barati RMSE':>12}")
for i, s in enumerate(SEQS):
    print(f"{s:>5}{mse_m[i].mean():>11.4f}{mse_b[i].mean():>12.4f}"
          f"{np.sqrt(mse_m[i].mean()):>11.4f}{np.sqrt(mse_b[i].mean()):>12.4f}")
print(f"\nOVERALL physical MSE: mine={mse_m.mean():.4f}  barati={mse_b.mean():.4f}  ratio={mse_m.mean()/mse_b.mean():.3f}")
print(f"OVERALL RMSE:         mine={np.sqrt(mse_m.mean()):.4f}  barati={np.sqrt(mse_b.mean()):.4f}")
print("(ratio < 1  =>  your model has lower error)")

In [ ]:
# --- per-frame MSE, both models, per sequence ---
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
for ax, i, s in zip(axes.ravel(), range(4), SEQS):
    ax.plot(mse_m[i], color="C0", label=f"mine ({mse_m[i].mean():.3f})")
    ax.plot(mse_b[i], color="C3", label=f"baratilab ({mse_b[i].mean():.3f})")
    ax.set_title(f"seq {s}"); ax.set_xlabel("frame"); ax.set_ylabel("physical MSE")
    ax.grid(alpha=0.3); ax.legend()
fig.suptitle("Per-frame reconstruction MSE (physical) — your model vs BaratiLab baseline", fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# --- mean MSE bar chart + per-frame advantage ---
x = np.arange(4); w = 0.35
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].bar(x - w/2, mse_m.mean(1), w, label="mine", color="C0")
ax[0].bar(x + w/2, mse_b.mean(1), w, label="baratilab", color="C3")
ax[0].set_xticks(x); ax[0].set_xticklabels([f"seq{s}" for s in SEQS])
ax[0].set_ylabel("mean physical MSE"); ax[0].set_title("mean MSE per sequence"); ax[0].legend()
diff = mse_b - mse_m    # >0 => mine better
for i, s in enumerate(SEQS):
    ax[1].plot(diff[i], label=f"seq{s}")
ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_title("MSE advantage (baratilab - mine);  >0 = your model better")
ax[1].set_xlabel("frame"); ax[1].grid(alpha=0.3); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
print(f"your model has lower MSE on {(diff > 0).mean()*100:.1f}% of all frames")

In [ ]:
# --- qualitative: input | GT | mine | baratilab | error maps ---
si, fr, ch = 0, 160, 1   # sequence index (0->seq36), frame, middle triplet channel
G = ref[si, fr, :, :, ch]; I = inp[si, fr, :, :, ch]
M = pred_m[si, fr, :, :, ch]; B = pred_b[si, fr, :, :, ch]
vmax = float(np.percentile(np.abs(G), 99)); emax = vmax * 0.6
panels = [("input (sparse+NN-fill)", I, False), ("ground truth", G, False),
          ("mine", M, False), ("baratilab", B, False),
          ("|mine - GT|", np.abs(M - G), True), ("|baratilab - GT|", np.abs(B - G), True)]
fig, axes = plt.subplots(1, 6, figsize=(22, 4))
for ax, (t, img, is_err) in zip(axes, panels):
    if is_err:
        ax.imshow(img, cmap="magma", vmin=0, vmax=emax)
    else:
        ax.imshow(img, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
    ax.set_title(t, fontsize=10); ax.axis("off")
fig.suptitle(f"seq {SEQS[si]}, frame {fr} (physical units)", fontsize=12)
plt.tight_layout(); plt.show()